In [ ]:
# Fine-Tuning ViT-B32 on MRI Classification with Full Evaluation

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy.special import softmax
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
import random
from transformers import AutoFeatureExtractor, ViTForImageClassification
from torch.cuda.amp import autocast, GradScaler

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Paths and Classes
train_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Training'
test_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Testing'
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
valid_classes = set(class_names)

# Feature extractor for ViT-B/32
extractor = AutoFeatureExtractor.from_pretrained('google/vit-base-patch32-224-in21k')

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=extractor.image_mean, std=extractor.image_std)
])

# Filter function
def filter_dataset(path, transform):
    dataset = datasets.ImageFolder(path, transform=transform)
    valid_indices = [i for i, (_, label) in enumerate(dataset.samples) if dataset.classes[label] in valid_classes]
    return Subset(dataset, valid_indices)

# Load datasets
train_ds_full = filter_dataset(train_path, transform)
test_ds = filter_dataset(test_path, transform)
val_size = int(0.2 * len(train_ds_full))
train_size = len(train_ds_full) - val_size
train_ds, val_ds = random_split(train_ds_full, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

# Load ViT-B/32 model
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch32-224-in21k',
    num_labels=4
)
model.to(device)

# Optimizer, Loss, Scaler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = GradScaler()

# Train function
def train(model, loader):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images).logits
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader, return_logits=False):
    model.eval()
    all_preds, all_labels, all_logits = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            if return_logits:
                all_logits.extend(outputs.cpu().numpy())
    if return_logits:
        return all_preds, all_labels, np.array(all_logits)
    return all_preds, all_labels

# Train loop
losses, f1s = [], []
for epoch in range(3):
    loss = train(model, train_loader)
    preds, labels = evaluate(model, val_loader)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Val Acc: {acc:.4f}, F1: {f1:.4f}")
    losses.append(loss)
    f1s.append(f1)

# Plot training progress
plt.plot(range(len(losses)), losses, label='Loss')
plt.plot(range(len(f1s)), f1s, label='F1 Score')
plt.xlabel('Epoch')
plt.ylabel('Metric')
plt.title('Training Progress')
plt.legend()
plt.show()

# Final evaluation on test set
preds, labels, logits = evaluate(model, test_loader, return_logits=True)
logits_proba = softmax(logits, axis=1)

print("\nTest Evaluation")
print("Accuracy:", accuracy_score(labels, preds))
print("F1 Score:", f1_score(labels, preds, average='macro'))
print("ROC AUC:", roc_auc_score(labels, logits_proba, multi_class='ovr'))
print(classification_report(labels, preds, target_names=class_names))

# Save outputs
os.makedirs('ovie results/reports', exist_ok=True)

# Confusion matrix
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
cm_path = 'ovie results/reports/confusion_vit_b32_matrix.png'
plt.savefig(cm_path)
plt.show()
print(f"✅ Confusion matrix saved at: {cm_path}")

# ROC curve for multiclass
plt.figure(figsize=(10, 8))
labels_bin = label_binarize(labels, classes=[0, 1, 2, 3])
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], logits_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {class_names[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True)
roc_path = 'ovie results/reports/roc_vit_curves.png'
plt.savefig(roc_path)
plt.show()
print(f"✅ ROC curves saved at: {roc_path}")

# Export metrics to Excel
metrics = {
    "Accuracy": [accuracy_score(labels, preds)],
    "F1 Score": [f1_score(labels, preds, average='macro')],
    "ROC AUC": [roc_auc_score(labels, logits_proba, multi_class='ovr')]
}
metrics_df = pd.DataFrame(metrics)
report_dict = classification_report(labels, preds, output_dict=True, target_names=class_names)
report_df = pd.DataFrame(report_dict).transpose()
excel_path = 'ovie results/reports/vit_b32_metrics_report.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="Scores")
    report_df.to_excel(writer, sheet_name="Classification Report")
print(f"\n✅ Metrics exported to: {excel_path}")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from scipy.special import softmax
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Final test evaluation
preds, labels, logits = evaluate(model, test_loader, return_logits=True)

# Convert logits to probabilities
logits_proba = softmax(logits, axis=1)

print("\nTest Evaluation")
print("Accuracy:", accuracy_score(labels, preds))
print("F1 Score:", f1_score(labels, preds, average='macro'))
print("ROC AUC:", roc_auc_score(labels, logits_proba, multi_class='ovr'))
print(classification_report(labels, preds, target_names=class_names, labels=[0, 1, 2, 3]))

# Confusion Matrix
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
import os
os.makedirs('ovie results/reports', exist_ok=True)  # Make sure the folder exists
# Save confusion matrix plot
cm_path = 'ovie results/reports/confusion_fvit_matrix.png'
plt.savefig(cm_path)
plt.show()

print(f"✅ Confusion matrix saved at: {cm_path}")

# ROC Curve plotting for multiclass (one-vs-rest)
plt.figure(figsize=(10, 8))

# Binarize the labels for ROC calculation
from sklearn.preprocessing import label_binarize
labels_bin = label_binarize(labels, classes=[0, 1, 2, 3])
n_classes = labels_bin.shape[1]

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], logits_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {class_names[i]} (area = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True)

# Save ROC curve plot
roc_path = 'ovie results/reports/roc_curves.png'
plt.savefig(roc_path)
plt.show()

print(f"✅ ROC curves saved at: {roc_path}")

# Export metrics
metrics = {
    "Accuracy": [accuracy_score(labels, preds)],
    "F1 Score": [f1_score(labels, preds, average='macro')],
    "ROC AUC": [roc_auc_score(labels, logits_proba, multi_class='ovr')]
}
metrics_df = pd.DataFrame(metrics)
report_dict = classification_report(labels, preds, output_dict=True, target_names=class_names, labels=[0, 1, 2, 3])
report_df = pd.DataFrame(report_dict).transpose()

excel_path = 'ovie results/reports/ftvt_i16_metrics_report.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="Scores")
    report_df.to_excel(writer, sheet_name="Classification Report")

print(f"\n✅ Metrics exported to: {excel_path}")


In [ ]:
# Fine-Tuning ViT-B32 on MRI Classification with Full Evaluation

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy.special import softmax
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
import random
from transformers import AutoFeatureExtractor, ViTForImageClassification
from torch.cuda.amp import autocast, GradScaler

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Paths and Classes
train_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Training'
test_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Testing'
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
valid_classes = set(class_names)

# Feature extractor for ViT-B/32
extractor = AutoFeatureExtractor.from_pretrained('google/vit-base-patch32-224-in21k')

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=extractor.image_mean, std=extractor.image_std)
])

# Filter function
def filter_dataset(path, transform):
    dataset = datasets.ImageFolder(path, transform=transform)
    valid_indices = [i for i, (_, label) in enumerate(dataset.samples) if dataset.classes[label] in valid_classes]
    return Subset(dataset, valid_indices)

# Load datasets
train_ds_full = filter_dataset(train_path, transform)
test_ds = filter_dataset(test_path, transform)
val_size = int(0.25 * len(train_ds_full))
train_size = len(train_ds_full) - val_size
train_ds, val_ds = random_split(train_ds_full, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

# Load ViT-B/32 model
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch32-224-in21k',
    num_labels=4
)
model.to(device)

# Optimizer, Loss, Scaler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = GradScaler()

# Train function
def train(model, loader):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images).logits
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader, return_logits=False):
    model.eval()
    all_preds, all_labels, all_logits = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            if return_logits:
                all_logits.extend(outputs.cpu().numpy())
    if return_logits:
        return all_preds, all_labels, np.array(all_logits)
    return all_preds, all_labels

# Train loop
losses, f1s = [], []
for epoch in range(25):
    loss = train(model, train_loader)
    preds, labels = evaluate(model, val_loader)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Val Acc: {acc:.4f}, F1: {f1:.4f}")
    losses.append(loss)
    f1s.append(f1)

# Plot training progress
plt.plot(range(len(losses)), losses, label='Loss')
plt.plot(range(len(f1s)), f1s, label='F1 Score')
plt.xlabel('Epoch')
plt.ylabel('Metric')
plt.title('Training Progress')
plt.legend()
plt.show()

# Final evaluation on test set
preds, labels, logits = evaluate(model, test_loader, return_logits=True)
logits_proba = softmax(logits, axis=1)

print("\nTest Evaluation")
print("Accuracy:", accuracy_score(labels, preds))
print("F1 Score:", f1_score(labels, preds, average='macro'))
print("ROC AUC:", roc_auc_score(labels, logits_proba, multi_class='ovr'))
print(classification_report(labels, preds, target_names=class_names))

# Save outputs
os.makedirs('ovie results/reports', exist_ok=True)

# Confusion matrix
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
cm_path = 'ovie results/reports/confusion_vit_b32_matrix.png'
plt.savefig(cm_path)
plt.show()
print(f"✅ Confusion matrix saved at: {cm_path}")

# ROC curve for multiclass
plt.figure(figsize=(10, 8))
labels_bin = label_binarize(labels, classes=[0, 1, 2, 3])
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], logits_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {class_names[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True)
roc_path = 'ovie results/reports/roc_vit_b32_curves.png'
plt.savefig(roc_path)
plt.show()
print(f"✅ ROC curves saved at: {roc_path}")

# Export metrics to Excel
metrics = {
    "Accuracy": [accuracy_score(labels, preds)],
    "F1 Score": [f1_score(labels, preds, average='macro')],
    "ROC AUC": [roc_auc_score(labels, logits_proba, multi_class='ovr')]
}
metrics_df = pd.DataFrame(metrics)
report_dict = classification_report(labels, preds, output_dict=True, target_names=class_names)
report_df = pd.DataFrame(report_dict).transpose()
excel_path = 'ovie results/reports/vit_b32_metrics_report.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="Scores")
    report_df.to_excel(writer, sheet_name="Classification Report")
print(f"\n✅ Metrics exported to: {excel_path}")
